# E1 -- double well (1D): plot notebook

**This notebook only reads saved results.**

It must not call a sampler, a PT tuner, an LSC quadrature refinement, a `dt` refinement, or a reference builder, and it must not recompute any official metric. Every number drawn here already exists in a run's `metrics_timeseries.csv` or `cost_timeseries.csv`, written by `E1_double_well_run.ipynb` at run time.

Scatter, CDF, histogram, and KDE panels are **display only**. They visualise the saved sample snapshots and **never override, correct, or stand in for** the numbers in `metrics_timeseries.csv`. If a picture and a saved metric disagree, the saved metric is the result.

Method colours, markers, and display names come from `configs/registry.yaml`; which runs to draw and how to lay them out comes from `configs/plots/manuscript.yaml`. Neither table is redefined here.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, "..")  # importable when launched from notebooks/

from src.catalog import select_runs
from src.plotting import (curve_figure, load_plot_config, load_runs,
                          save_figure, snapshot_figure)

REPO_ROOT = Path("..")
EXPERIMENT_ID = "E1"

plot_config = load_plot_config(REPO_ROOT / "configs" / "plots" / "manuscript.yaml")
defaults = plot_config["defaults"]
spec = plot_config[EXPERIMENT_ID]
figures = spec["figures"]

EXPERIMENT_DIR = REPO_ROOT / "results" / spec["experiment_key"]
OUTPUT_DIR = REPO_ROOT / defaults["output_root"] / spec["experiment_key"]
FORMATS = tuple(defaults["output_formats"])  # png, pdf, svg, tiff
FIGURES = {}


def load_available(experiment_dir, spec, **kwargs):
    """Load the spec's runs, tolerating methods this campaign could not run.

    Canonical (untamed) variants are expected to be unusable on several of
    these targets: the drift is not truncated, so at the step size the run
    stage settled on they are genuinely unstable. That is a result, not a plotting problem, and a
    figure must still draw the methods that did run. Uncalibratable methods are
    annotated by the plotting module; anything still missing here is dropped
    with a printed warning rather than aborting the notebook.
    """
    try:
        return load_runs(experiment_dir, spec, **kwargs)
    except ValueError as error:
        if "requires missing methods" not in str(error):
            raise
        print(f"warning: {error}")
        print("plotting only the runs that completed")
        # methods=None disables the all-methods-required check; the spec's
        # variant filters still exclude unrelated methods.
        return load_runs(experiment_dir, spec, methods=None, **kwargs)


print(f"{EXPERIMENT_ID}: {len(figures)} specified figures -> {OUTPUT_DIR}")

E1: 3 specified figures -> ../figures/E1_double_well


## What is plottable

Load the derived catalog (rebuilding it from the manifests if it is missing) and list the runs it admits, so it is visible up front which methods, variants, and step sizes this notebook can actually draw. Nothing is run here; this is a directory listing.

In [2]:
runs = select_runs(EXPERIMENT_DIR, latest_only=defaults["latest_run_only"])

print(f"{len(runs)} plottable runs\n")
print(f"{'method':<12}{'variant label':<34}{'tame':<7}{'dt':<10}run id")
for row in runs:
    print(f"{row['method']:<12}{row['variant_label']:<34}"
          f"{str(row['tame']):<7}{str(row['dt']):<10}{row['run_id']}")

18 plottable runs

method      variant label                     tame   dt        run id
FLA         FLA alpha=1.6, canonical          False  0.0025    FLA-alpha1.6-canonical-dt0.0025-20260806T212433277817Z
FLA         FLA alpha=1.6, tamed              True   0.005     FLA-alpha1.6-tamed-dt0.005-20260806T212442427890Z
FLA         FLA alpha=1.7, canonical          False  0.0025    FLA-alpha1.7-canonical-dt0.0025-20260806T212455414491Z
FLA         FLA alpha=1.7, tamed              True   0.005     FLA-alpha1.7-tamed-dt0.005-20260806T212504526770Z
FLA         FLA alpha=1.8, canonical          False  0.0025    FLA-alpha1.8-canonical-dt0.0025-20260806T212516474798Z
FLA         FLA alpha=1.8, tamed              True   0.005     FLA-alpha1.8-tamed-dt0.005-20260806T212525615638Z
LSC-CP      LSC-CP, tamed                     True   0.005     LSC-CP-tamed-dt0.005-20260806T213100498757Z
LSC-CP-RA   LSC-CP-RA (A=4), tamed            True   0.005     LSC-CP-RA-A4-tamed-dt0.005-20260806T213251466645

### Figure E1.1 -- CDFs and the double-well potential

One axis with **twin y-axes**, not two panels. The x axis is position.

* Left y-axis: cumulative distribution functions -- the **exact target in black**, plus ULA, ULD, Raw-CP, and LSC-CP, all at one matched simulation time.
* Right y-axis: the double-well potential, drawn in grey at low alpha **behind** the CDF curves.

The legend says **ULD**. It never says BAOAB.

The CDF curves are display only: they visualise saved samples and do not override the KS distance or the exact one-dimensional $W_2$ already recorded in `metrics_timeseries.csv`.

In [3]:
figure_spec = figures["E1.1_cdf_and_potential"]

FIGURES["E1.1_cdf_and_potential"] = snapshot_figure(
    load_available(EXPERIMENT_DIR, figure_spec), figure_spec)
FIGURES["E1.1_cdf_and_potential"]

<Figure size 480x330 with 2 Axes>

### Figure E1.2 -- nonlocal efficiency

A 3x2 grid. **Rows** are the three primary metrics: the exact one-dimensional $W_2$, MMD$^2$, and the KS distance. **Columns** are the two official x axes: against simulation time, and against force-equivalent cost (FEE) per particle. Methods are limited to **FLA, LSC-CP, and LSC-CP-RA**.

In [4]:
figure_spec = figures["E1.2_nonlocal_efficiency"]

FIGURES["E1.2_nonlocal_efficiency"] = curve_figure(
    load_available(EXPERIMENT_DIR, figure_spec), figure_spec)
FIGURES["E1.2_nonlocal_efficiency"]

<Figure size 740x750 with 6 Axes>

### Figure E1.3 -- LSC score potential-evaluation cost

An **LSC-only** figure, comparing full deterministic-quadrature LSC-CP against LSC-CP-RA(A) and nothing else.

The x axis counts **LSC score potential evaluations only**. It is not a complete computational cost, and no method with a zero extra-potential count may appear on it.

In [5]:
figure_spec = figures["E1.3_lsc_score_cost"]

FIGURES["E1.3_lsc_score_cost"] = curve_figure(
    load_available(EXPERIMENT_DIR, figure_spec), figure_spec)
FIGURES["E1.3_lsc_score_cost"]

<Figure size 400x530 with 2 Axes>

## Canonical, tamed, and paired views

The main curve figure is regenerated three times, from the same saved runs: **canonical only**, **tamed only**, and the **paired** canonical-versus-tamed overlay.

The convention, taken from `configs/registry.yaml` and the plot defaults:

* the **method** sets the **colour**, and taming never changes it;
* **canonical** is a **solid** line;
* **tamed** is a **dashed** line;
* **hyperparameter values** are distinguished by **marker**, and the value is written into the legend label.

So colour answers "which method", line style answers "tamed or not", and marker answers "which hyperparameter value".

In [6]:
MAIN_CURVE_FIGURE = "E1.2_nonlocal_efficiency"

for view in defaults["tame_views"]:
    view_spec = {**figures[MAIN_CURVE_FIGURE], "tame_view": view}
    FIGURES[f"{MAIN_CURVE_FIGURE}__{view}"] = curve_figure(
        load_available(EXPERIMENT_DIR, view_spec), view_spec)

print("tame views:", list(defaults["tame_views"]))

tame views: ['canonical_only', 'tamed_only', 'paired']


## Export

Every figure built above is written to **PNG, PDF, SVG, and TIFF** under `figures/<experiment key>/`, from the format list in the plot defaults. Re-running this cell overwrites the files in place; it never touches anything under `results/`.

In [7]:
for name, figure in FIGURES.items():
    save_figure(figure, name, OUTPUT_DIR, formats=FORMATS)
    print(f"saved {name}  [{', '.join(FORMATS)}]")

print(f"\n{len(FIGURES)} figures written under {OUTPUT_DIR}")

saved E1.1_cdf_and_potential  [png, pdf, svg, tiff]


saved E1.2_nonlocal_efficiency  [png, pdf, svg, tiff]


saved E1.3_lsc_score_cost  [png, pdf, svg, tiff]


saved E1.2_nonlocal_efficiency__canonical_only  [png, pdf, svg, tiff]


saved E1.2_nonlocal_efficiency__tamed_only  [png, pdf, svg, tiff]


saved E1.2_nonlocal_efficiency__paired  [png, pdf, svg, tiff]

6 figures written under ../figures/E1_double_well
